# Cocoon Batch Video Pipeline — Wav2Lip + FFmpeg

Notebook này chạy pipeline local trên Colab trước khi mở ngrok/API.

Input chính:

```text
/content/drive/MyDrive/cocoon_ai_video/
  data/
    master_script.json
    outputs/
      cocoon_test_001_tts_package/
        audio/
          S001.wav
          S002.wav
          ...
    video_templates/
      S001.mp4                    # ưu tiên cao nhất nếu có
      S002.mp4
      HOST_TALK.mp4               # fallback theo scene_type
      CTA.mp4
      FAQ_ANSWER.mp4
      HOST_PHONE_READING.mp4
      PRODUCT_CLOSEUP.mp4
    scene_images/                 # optional fallback nếu thiếu video product
      S003.png
      S006.png
      ...
  models/
    Wav2Lip-SD-GAN.pt
```

Output:

```text
/content/drive/MyDrive/cocoon_ai_video/
  data/outputs/cocoon_test_001_video/
    scenes/S001.mp4 ...
    final/A_MAIN_SALES_LOOP.mp4
    final/B_COMMENT_READING_LOOP.mp4
    final/C_CTA_LOOP.mp4
    final/cocoon_test_001_FULL_LOOP.mp4
    cocoon_test_001_video_package.zip
```

Logic:

- `needs_lipsync = true` → dùng Wav2Lip với `video template + audio`.
- `needs_lipsync = false` → không chạy Wav2Lip, chỉ loop/trim visual + mux audio + overlay.
- Ưu tiên video riêng theo `scene_id`, nếu không có thì fallback theo `scene_type`.
- Nếu scene không cần lip-sync và thiếu video, có thể dùng ảnh trong `scene_images/` để tạo pseudo-video bằng FFmpeg.


## 1. Mount Drive + kiểm tra GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!nvidia-smi
!ffmpeg -version | head -n 3


## 2. Cấu hình path

Sửa `PROJECT_DIR` nếu project của bạn để ở folder khác.


In [ ]:
from pathlib import Path
import os
import json
import shutil
import subprocess
import csv
import textwrap
from datetime import datetime

# ===== ROOT CONFIG =====
PROJECT_DIR = Path("/content/drive/MyDrive/cocoon_ai_video")

DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"

SCRIPT_PATH = DATA_DIR / "master_script.json"

# Audio đã tạo từ OmniVoice / TTS
AUDIO_DIR = DATA_DIR / "outputs" / "cocoon_test_001_tts_package" / "audio"

# Video/ảnh đầu vào
VIDEO_TEMPLATE_DIR = DATA_DIR / "video_templates"
SCENE_IMAGE_DIR = DATA_DIR / "scene_images"

# Wav2Lip
WAV2LIP_DIR = Path("/content/Wav2Lip")
CHECKPOINT_SRC = MODEL_DIR / "Wav2Lip-SD-GAN.pt"
CHECKPOINT_DST = WAV2LIP_DIR / "checkpoints" / "Wav2Lip-SD-GAN.pt"

# Output
JOB_ID_FALLBACK = "cocoon_test_001"
OUTPUT_ROOT = DATA_DIR / "outputs" / f"{JOB_ID_FALLBACK}_video"
WORK_DIR = OUTPUT_ROOT / "_work"
SCENE_OUT_DIR = OUTPUT_ROOT / "scenes"
FINAL_OUT_DIR = OUTPUT_ROOT / "final"
REPORT_DIR = OUTPUT_ROOT / "reports"

# Render config
TARGET_W = 1080
TARGET_H = 1920
FPS = 25
CRF = 18
PRESET = "veryfast"

# Wav2Lip config
PADS = ["0", "20", "0", "0"]
RESIZE_FACTOR = "2"

# Nếu True: cho phép dùng ảnh host/ảnh scene để chạy lipsync khi thiếu video.
# Khuyến nghị False nếu muốn motion tự nhiên.
ALLOW_STATIC_IMAGE_FOR_LIPSYNC = False

for p in [DATA_DIR, MODEL_DIR, VIDEO_TEMPLATE_DIR, SCENE_IMAGE_DIR, WORK_DIR, SCENE_OUT_DIR, FINAL_OUT_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("SCRIPT_PATH:", SCRIPT_PATH)
print("AUDIO_DIR:", AUDIO_DIR)
print("VIDEO_TEMPLATE_DIR:", VIDEO_TEMPLATE_DIR)
print("SCENE_IMAGE_DIR:", SCENE_IMAGE_DIR)
print("CHECKPOINT_SRC:", CHECKPOINT_SRC)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


## 3. Setup Wav2Lip

Cell này clone Wav2Lip, cài dependency, tải face detector `s3fd.pth`, và copy checkpoint từ Drive.

Bạn cần đặt checkpoint ở:

```text
/content/drive/MyDrive/cocoon_ai_video/models/Wav2Lip-SD-GAN.pt
```

Nếu bạn dùng checkpoint khác, sửa `CHECKPOINT_SRC` ở cell cấu hình.


In [ ]:
# Clone Wav2Lip nếu chưa có
if not WAV2LIP_DIR.exists():
    !git clone https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip
else:
    print("Wav2Lip already exists:", WAV2LIP_DIR)

# Cài dependency
# Lưu ý: Colab đôi khi có xung đột tensorflow/librosa, nên ép librosa==0.9.2 giống mẫu quick trial.
!pip uninstall -y tensorflow tensorflow-gpu >/dev/null 2>&1 || true
!cd /content/Wav2Lip && pip install -q -r requirements.txt
!pip uninstall -y librosa >/dev/null 2>&1 || true
!pip install -q librosa==0.9.2 opencv-python-headless tqdm

# Tải face detector
s3fd_path = WAV2LIP_DIR / "face_detection" / "detection" / "sfd" / "s3fd.pth"
s3fd_path.parent.mkdir(parents=True, exist_ok=True)
if not s3fd_path.exists():
    !wget -q "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth" -O "/content/Wav2Lip/face_detection/detection/sfd/s3fd.pth"
else:
    print("s3fd already exists:", s3fd_path)

# Copy checkpoint
CHECKPOINT_DST.parent.mkdir(parents=True, exist_ok=True)
if CHECKPOINT_SRC.exists():
    shutil.copy2(CHECKPOINT_SRC, CHECKPOINT_DST)
    print("Copied checkpoint:", CHECKPOINT_DST)
elif CHECKPOINT_DST.exists():
    print("Checkpoint already exists:", CHECKPOINT_DST)
else:
    raise FileNotFoundError(
        f"Không thấy checkpoint. Đặt file tại {CHECKPOINT_SRC} "
        "hoặc sửa CHECKPOINT_SRC trong cell cấu hình."
    )

print("Setup done.")


## 4. Load JSON + validate assets

Cell này đọc `master_script.json`, kiểm tra audio theo `scene_id`, kiểm tra visual template theo thứ tự:

1. `video_templates/{scene_id}.mp4`
2. `video_templates/{scene_type}.mp4`
3. `scene_images/{scene_id}.png|jpg|jpeg|webp`
4. `scene_images/{scene_type}.png|jpg|jpeg|webp`

Với scene cần lip-sync, notebook sẽ ưu tiên video. Ảnh tĩnh cho lip-sync chỉ chạy nếu `ALLOW_STATIC_IMAGE_FOR_LIPSYNC=True`.


In [ ]:
def load_script(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Không thấy master_script.json tại: {path}")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if "scenes" not in data:
        raise ValueError("JSON thiếu field 'scenes'.")
    if "playlist" not in data:
        raise ValueError("JSON thiếu field 'playlist'.")
    return data

script = load_script(SCRIPT_PATH)
JOB_ID = script.get("job_id", JOB_ID_FALLBACK)

# Update output theo job_id thật
OUTPUT_ROOT = DATA_DIR / "outputs" / f"{JOB_ID}_video"
WORK_DIR = OUTPUT_ROOT / "_work"
SCENE_OUT_DIR = OUTPUT_ROOT / "scenes"
FINAL_OUT_DIR = OUTPUT_ROOT / "final"
REPORT_DIR = OUTPUT_ROOT / "reports"

for p in [WORK_DIR, SCENE_OUT_DIR, FINAL_OUT_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

scenes = sorted(script["scenes"], key=lambda s: s.get("order", 999999))
playlists = script["playlist"]

print("JOB_ID:", JOB_ID)
print("Scenes:", len(scenes))
print("Playlists:", [p["clip_id"] for p in playlists])
print("Output:", OUTPUT_ROOT)


In [ ]:
IMAGE_EXTS = [".png", ".jpg", ".jpeg", ".webp"]
VIDEO_EXTS = [".mp4", ".mov", ".mkv", ".webm"]

def candidate_video_paths(scene):
    scene_id = scene["scene_id"]
    scene_type = scene.get("scene_type", "")
    candidates = []

    for ext in VIDEO_EXTS:
        candidates.append(VIDEO_TEMPLATE_DIR / f"{scene_id}{ext}")

    if scene_type:
        for ext in VIDEO_EXTS:
            candidates.append(VIDEO_TEMPLATE_DIR / f"{scene_type}{ext}")

    return candidates

def candidate_image_paths(scene):
    scene_id = scene["scene_id"]
    scene_type = scene.get("scene_type", "")
    candidates = []

    for ext in IMAGE_EXTS:
        candidates.append(SCENE_IMAGE_DIR / f"{scene_id}{ext}")

    if scene_type:
        for ext in IMAGE_EXTS:
            candidates.append(SCENE_IMAGE_DIR / f"{scene_type}{ext}")

    return candidates

def find_visual_asset(scene):
    needs_lipsync = bool(scene.get("needs_lipsync", False))

    for p in candidate_video_paths(scene):
        if p.exists():
            return {"path": p, "kind": "video"}

    for p in candidate_image_paths(scene):
        if p.exists():
            if needs_lipsync and not ALLOW_STATIC_IMAGE_FOR_LIPSYNC:
                continue
            return {"path": p, "kind": "image"}

    return None

def validate_assets(scenes):
    rows = []
    missing = []

    for scene in scenes:
        scene_id = scene["scene_id"]
        audio_path = AUDIO_DIR / f"{scene_id}.wav"
        visual = find_visual_asset(scene)

        row = {
            "scene_id": scene_id,
            "order": scene.get("order"),
            "scene_type": scene.get("scene_type"),
            "needs_lipsync": scene.get("needs_lipsync"),
            "audio_path": str(audio_path),
            "audio_exists": audio_path.exists(),
            "visual_path": str(visual["path"]) if visual else "",
            "visual_kind": visual["kind"] if visual else "",
            "overlay_text": scene.get("overlay_text") or "",
        }
        rows.append(row)

        if not audio_path.exists() or visual is None:
            missing.append(row)

    report_path = REPORT_DIR / "asset_report.csv"
    with open(report_path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    return rows, missing, report_path

asset_rows, missing_assets, asset_report_path = validate_assets(scenes)

print("Asset report:", asset_report_path)
print("Missing count:", len(missing_assets))

if missing_assets:
    print("\nMissing / blocked assets:")
    for r in missing_assets:
        print(
            f"- {r['scene_id']} | audio={r['audio_exists']} | "
            f"visual={bool(r['visual_path'])} | type={r['scene_type']} | "
            f"needs_lipsync={r['needs_lipsync']}"
        )
else:
    print("All assets are ready.")


## 5. FFmpeg + Wav2Lip helper functions

In [ ]:
def run(cmd, cwd=None, check=True):
    """Run shell command with visible logs."""
    cmd = [str(x) for x in cmd]
    print("\n$ " + " ".join(cmd))
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    print(result.stdout[-4000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}: {' '.join(cmd)}")
    return result

def ffprobe_duration(path: Path) -> float:
    result = subprocess.check_output([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path)
    ])
    return float(result.decode("utf-8").strip())

def normalize_video_to_audio(template_video: Path, audio_path: Path, out_path: Path):
    """Loop/trim video theo audio duration, chuẩn hóa 9:16 1080x1920, 25fps, no audio."""
    dur = ffprobe_duration(audio_path)

    vf = (
        f"scale={TARGET_W}:{TARGET_H}:force_original_aspect_ratio=increase,"
        f"crop={TARGET_W}:{TARGET_H},"
        f"fps={FPS},"
        "format=yuv420p"
    )

    run([
        "ffmpeg", "-y",
        "-stream_loop", "-1",
        "-i", template_video,
        "-t", f"{dur:.3f}",
        "-an",
        "-vf", vf,
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-pix_fmt", "yuv420p",
        out_path
    ])

    return out_path

def image_to_motion_video(image_path: Path, audio_path: Path, out_path: Path):
    """Tạo pseudo-video từ ảnh theo audio duration bằng zoompan rất nhẹ."""
    dur = ffprobe_duration(audio_path)
    frames = max(1, int(dur * FPS))

    vf = (
        f"scale={TARGET_W}:{TARGET_H}:force_original_aspect_ratio=increase,"
        f"crop={TARGET_W}:{TARGET_H},"
        f"zoompan=z='min(zoom+0.0008,1.06)':d={frames}:s={TARGET_W}x{TARGET_H}:fps={FPS},"
        "format=yuv420p"
    )

    run([
        "ffmpeg", "-y",
        "-loop", "1",
        "-i", image_path,
        "-t", f"{dur:.3f}",
        "-vf", vf,
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-pix_fmt", "yuv420p",
        out_path
    ])

    return out_path

def mux_audio(video_path: Path, audio_path: Path, out_path: Path):
    """Gắn audio vào video, trim theo track ngắn hơn."""
    run([
        "ffmpeg", "-y",
        "-i", video_path,
        "-i", audio_path,
        "-map", "0:v:0",
        "-map", "1:a:0",
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-c:a", "aac",
        "-ar", "44100",
        "-b:a", "192k",
        "-shortest",
        "-pix_fmt", "yuv420p",
        out_path
    ])
    return out_path

def add_overlay_text(video_path: Path, text: str, out_path: Path):
    """Thêm overlay text bằng drawtext textfile để giữ dấu tiếng Việt ổn hơn."""
    if not text:
        shutil.copy2(video_path, out_path)
        return out_path

    text_file = WORK_DIR / f"{out_path.stem}_overlay.txt"
    text_file.write_text(str(text), encoding="utf-8")

    # DejaVuSans có sẵn trong Colab và hỗ trợ tiếng Việt khá ổn.
    font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"

    draw = (
        f"drawtext=fontfile={font_path}:"
        f"textfile={text_file}:"
        "fontcolor=white:"
        "fontsize=42:"
        "line_spacing=10:"
        "box=1:"
        "boxcolor=black@0.58:"
        "boxborderw=24:"
        "x=(w-text_w)/2:"
        "y=h-text_h-180"
    )

    run([
        "ffmpeg", "-y",
        "-i", video_path,
        "-vf", draw,
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-c:a", "copy",
        "-pix_fmt", "yuv420p",
        out_path
    ])

    return out_path

def run_wav2lip(face_video: Path, audio_path: Path, out_path: Path):
    """Chạy Wav2Lip. Tool mặc định ghi Wav2Lip/results/result_voice.mp4."""
    result_path = WAV2LIP_DIR / "results" / "result_voice.mp4"
    if result_path.exists():
        result_path.unlink()

    run([
        "python", "inference.py",
        "--checkpoint_path", CHECKPOINT_DST,
        "--face", face_video,
        "--audio", audio_path,
        "--pads", *PADS,
        "--resize_factor", RESIZE_FACTOR,
    ], cwd=WAV2LIP_DIR)

    if not result_path.exists():
        raise RuntimeError("Wav2Lip không tạo ra Wav2Lip/results/result_voice.mp4")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(result_path), str(out_path))
    return out_path


## 6. Generate từng scene

Cell này tạo `scenes/Sxxx.mp4`.

Nếu scene lỗi, lỗi sẽ được ghi vào `reports/scene_errors.json`, pipeline vẫn chạy tiếp các scene khác.


In [ ]:
generated_scenes = {}
scene_logs = []
scene_errors = []

for scene in scenes:
    scene_id = scene["scene_id"]
    scene_type = scene.get("scene_type", "")
    needs_lipsync = bool(scene.get("needs_lipsync", False))
    overlay_text = scene.get("overlay_text")

    print("\n" + "=" * 90)
    print(f"SCENE {scene_id} | type={scene_type} | lipsync={needs_lipsync}")
    print("=" * 90)

    audio_path = AUDIO_DIR / f"{scene_id}.wav"
    visual = find_visual_asset(scene)

    if not audio_path.exists():
        msg = f"Missing audio: {audio_path}"
        print("SKIP:", msg)
        scene_errors.append({"scene_id": scene_id, "error": msg})
        continue

    if visual is None:
        msg = (
            f"Missing visual asset for {scene_id}. "
            f"Need video_templates/{scene_id}.mp4 or video_templates/{scene_type}.mp4. "
            f"For non-lipsync scenes, image fallback is scene_images/{scene_id}.png/jpg/webp."
        )
        print("SKIP:", msg)
        scene_errors.append({"scene_id": scene_id, "error": msg})
        continue

    try:
        asset_path = visual["path"]
        asset_kind = visual["kind"]

        base_video = WORK_DIR / f"{scene_id}_base.mp4"
        raw_scene = WORK_DIR / f"{scene_id}_raw.mp4"
        final_scene = SCENE_OUT_DIR / f"{scene_id}.mp4"

        if asset_kind == "video":
            normalize_video_to_audio(asset_path, audio_path, base_video)
        elif asset_kind == "image":
            image_to_motion_video(asset_path, audio_path, base_video)
        else:
            raise ValueError(f"Unsupported visual kind: {asset_kind}")

        if needs_lipsync:
            run_wav2lip(base_video, audio_path, raw_scene)
        else:
            mux_audio(base_video, audio_path, raw_scene)

        add_overlay_text(raw_scene, overlay_text, final_scene)

        generated_scenes[scene_id] = final_scene

        scene_logs.append({
            "scene_id": scene_id,
            "scene_type": scene_type,
            "needs_lipsync": needs_lipsync,
            "audio_path": str(audio_path),
            "visual_path": str(asset_path),
            "visual_kind": asset_kind,
            "output_path": str(final_scene),
            "status": "ok"
        })

        print("DONE:", final_scene)

    except Exception as e:
        print("ERROR:", repr(e))
        scene_errors.append({"scene_id": scene_id, "error": repr(e)})

# Save logs
manifest_path = REPORT_DIR / "generated_scene_manifest.csv"
if scene_logs:
    with open(manifest_path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(scene_logs[0].keys()))
        writer.writeheader()
        writer.writerows(scene_logs)

errors_path = REPORT_DIR / "scene_errors.json"
with open(errors_path, "w", encoding="utf-8") as f:
    json.dump(scene_errors, f, ensure_ascii=False, indent=2)

print("\nGenerated scenes:", len(generated_scenes))
print("Scene manifest:", manifest_path)
print("Errors:", errors_path)
scene_errors[:5]


## 7. Concat playlist thành từng loop

Cell này tạo:

- `A_MAIN_SALES_LOOP.mp4`
- `B_COMMENT_READING_LOOP.mp4`
- `C_CTA_LOOP.mp4`

Sau đó tạo `cocoon_test_001_FULL_LOOP.mp4`.


In [ ]:
def concat_videos(video_paths, out_path: Path):
    """Concat bằng demuxer + re-encode để tránh lỗi mismatch codec/timebase."""
    if not video_paths:
        raise ValueError(f"No videos to concat for {out_path}")

    concat_file = WORK_DIR / f"{out_path.stem}_concat.txt"

    lines = []
    for p in video_paths:
        p = Path(p).resolve()
        lines.append(f"file '{p}'")

    concat_file.write_text("\n".join(lines), encoding="utf-8")

    run([
        "ffmpeg", "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", concat_file,
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-c:a", "aac",
        "-ar", "44100",
        "-b:a", "192k",
        "-pix_fmt", "yuv420p",
        out_path
    ])

    return out_path

loop_outputs = []

for playlist in playlists:
    clip_id = playlist["clip_id"]
    scene_ids = playlist.get("scenes", [])

    paths = []
    missing = []

    for sid in scene_ids:
        if sid in generated_scenes and Path(generated_scenes[sid]).exists():
            paths.append(generated_scenes[sid])
        else:
            missing.append(sid)

    print("\nPlaylist:", clip_id)
    print("Included:", [Path(p).name for p in paths])
    if missing:
        print("Missing scenes:", missing)

    if not paths:
        print("SKIP playlist with no generated scenes:", clip_id)
        continue

    out_path = FINAL_OUT_DIR / f"{clip_id}.mp4"
    concat_videos(paths, out_path)
    loop_outputs.append(out_path)
    print("LOOP DONE:", out_path)

# Full loop
if loop_outputs:
    full_video = FINAL_OUT_DIR / f"{JOB_ID}_FULL_LOOP.mp4"
    concat_videos(loop_outputs, full_video)
    print("\nFINAL VIDEO:", full_video)
else:
    full_video = None
    print("No loop outputs generated.")


## 8. Preview kết quả trong Colab

In [ ]:
from IPython.display import Video, display

if full_video and Path(full_video).exists():
    display(Video(str(full_video), embed=True, width=360))
else:
    print("Chưa có full video để preview.")


## 9. Zip output để tải về

In [ ]:
zip_base = DATA_DIR / "outputs" / f"{JOB_ID}_video_package"
zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(OUTPUT_ROOT)
)

print("ZIP:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Không auto-download được, tải thủ công tại:", zip_path)
    print("Error:", repr(e))


## 10. Optional: tạo manifest JSON cho backend/ngrok sau này

Cell này không mở ngrok. Nó chỉ tạo file manifest để API/ngrok đọc sau.


In [ ]:
backend_manifest = {
    "job_id": JOB_ID,
    "created_at": datetime.utcnow().isoformat() + "Z",
    "output_root": str(OUTPUT_ROOT),
    "scene_dir": str(SCENE_OUT_DIR),
    "final_dir": str(FINAL_OUT_DIR),
    "full_video": str(full_video) if full_video else None,
    "loops": [str(p) for p in loop_outputs],
    "reports": {
        "asset_report": str(asset_report_path),
        "scene_manifest": str(REPORT_DIR / "generated_scene_manifest.csv"),
        "errors": str(REPORT_DIR / "scene_errors.json"),
    }
}

manifest_json_path = OUTPUT_ROOT / "video_pipeline_manifest.json"
with open(manifest_json_path, "w", encoding="utf-8") as f:
    json.dump(backend_manifest, f, ensure_ascii=False, indent=2)

print("Backend manifest:", manifest_json_path)
backend_manifest


## Ghi chú folder visual

Bạn có thể dùng một trong hai cách:

### Cách 1 — video riêng từng scene, tốt nhất

```text
data/video_templates/S001.mp4
data/video_templates/S002.mp4
...
data/video_templates/S019.mp4
```

### Cách 2 — video fallback theo scene_type

```text
data/video_templates/HOST_TALK.mp4
data/video_templates/CTA.mp4
data/video_templates/HOST_PHONE_READING.mp4
data/video_templates/FAQ_ANSWER.mp4
data/video_templates/PRODUCT_CLOSEUP.mp4
```

### Cách 3 — ảnh cho product closeup

Chỉ nên dùng cho scene `needs_lipsync=false`.

```text
data/scene_images/S003.png
data/scene_images/S006.png
data/scene_images/S009.png
data/scene_images/S012.png
```

Nếu muốn dùng ảnh host để lip-sync, đổi:

```python
ALLOW_STATIC_IMAGE_FOR_LIPSYNC = True
```

Nhưng chất lượng motion sẽ kém hơn video mẫu.
